In [1]:
import pyautogui
import time
import pandas as pd
from PIL import Image
import pytesseract
import os
import glob

# Load the Excel file
file_path = "C:\\Users\\seres\\Downloads\\test.xlsx"  # Update this path with your actual file path
sheet_name = "Seek"
df = pd.read_excel(file_path, sheet_name=sheet_name)

print(df.columns)
# Temp file to track progress and store results
progress_file = "C:\\Users\\seres\\Downloads\\progress.csv"  # You can use a different location or name if needed

# Load progress from the temp file
if os.path.exists(progress_file):
    progress_df = pd.read_csv(progress_file)
    processed_caseids = progress_df['caseid'].tolist()
else:
    # If the progress file does not exist, initialize an empty DataFrame
    progress_df = pd.DataFrame(columns=['caseid', 'Seeker'])
    processed_caseids = []

# Start processing from the next unprocessed row
start_index = df[~df['caseid'].isin(processed_caseids)].index[0]

time.sleep(5)  # Wait for the user to get ready or focus on the correct window

# Directory where screenshots are saved
screenshot_dir = "C:\\Users\\seres\\Documents\\ShareX\\Screenshots\\2024-08"

# Function to get the latest screenshot filename
def get_latest_screenshot():
    files = glob.glob(os.path.join(screenshot_dir, "TDS50E02_*.png"))
    if not files:
        return None
    # Sort files by creation time (newest first)
    return max(files, key=os.path.getctime)

# Coordinates to crop the region (left, top, right, bottom)
coordinates = (117, 504, 176, 529)

# Iterate through the rows in the DataFrame
for index, row in df.iloc[start_index:].iterrows():
    caseid = row['caseid']

    # Skip already processed rows
    if caseid in processed_caseids:
        continue

    # Convert numbers to integers before writing (if applicable)
    age = str(int(row['Age'])) if pd.notna(row['Age']) else ""
    ageday = str(int(row['AgeDay'])) if pd.notna(row['AgeDay']) else ""
    sex = str(int(row['Sex'])) if pd.notna(row['Sex']) else ""
    disc_type = str(int(row['Disc Type'])) if pd.notna(row['Disc Type']) else ""
    losd = str(int(row['LOSD'])) if pd.notna(row['LOSD']) else ""

    # Handle LOSHr as a time object and format it to "HHMM"
    if pd.notna(row['LOSHr']):
        loshr_time = row['LOSHr']
        loshr = loshr_time.strftime("%H%M")
    else:
        loshr = ""

    pdx = str(row['PDx']) if pd.notna(row['PDx']) else ""

    # Map columns to input fields
    pyautogui.write(age)
    if len(age) < 3:
        pyautogui.press('tab')
    pyautogui.write(ageday)
    if len(ageday) < 3:
        pyautogui.press('tab')
    pyautogui.write(sex)
    # pyautogui.press('tab')
    pyautogui.write(disc_type)
    pyautogui.press('tab')
    pyautogui.write(losd)
    if len(losd) < 4:
        pyautogui.press('tab')
    pyautogui.write(loshr)
    # pyautogui.press('tab')
    pyautogui.write(pdx)
    if len(pdx) < 6:
        pyautogui.press('tab')

    # Input secondary diagnoses
    for i in range(1, 13):
        sdx_col = f'SDx{i}'
        if pd.notna(row[sdx_col]):
            pyautogui.write(str(row[sdx_col]))
            if len(str(row[sdx_col])) < 6:
              pyautogui.press('tab')
        else:
            pyautogui.press('tab')

    # Input procedures
    for i in range(1, 21):
        proc_col = f'Proc{i}'
        if pd.notna(row[proc_col]):
            proc_value = str(int(row[proc_col]))  # Convert to integer to remove .0
            pyautogui.write(proc_value)
            if len(proc_value) < 8:
              pyautogui.press('tab')
        else:
            pyautogui.press('tab')
        
    # After filling all fields, simulate clicking the "Show DRG" button
    pyautogui.press('enter')

    # Add delay to wait for the result
    time.sleep(0.1)

    pyautogui.press('printscreen')

    time.sleep(0.1)
    # Get the latest screenshot
    latest_screenshot = get_latest_screenshot()
    if latest_screenshot:
        try:
            # Open the screenshot and crop it
            image = Image.open(latest_screenshot)
            cropped_image = image.crop(coordinates)
            
            # Perform OCR on the cropped image
            extracted_text = pytesseract.image_to_string(cropped_image, config='--psm 6').strip()
            print(f"OCR result for caseid {caseid}: {extracted_text}")  # Debugging statement

        except Exception as e:
            extracted_text = f"Error: {e}"
            print(f"OCR Error for caseid {caseid}: {e}")

    else:
        extracted_text = "No screenshot found"
        print(f"No screenshot found for caseid {caseid}")

    # Append the result to the progress DataFrame
    progress_df = pd.concat([progress_df, pd.DataFrame([[caseid, extracted_text]], columns=['caseid', 'Seeker'])], ignore_index=True)

    # Save the progress to the progress CSV file after each row
    progress_df.to_csv(progress_file, index=False)

    # Add a small delay to ensure actions are properly processed
    time.sleep(0.1)

    pyautogui.press('tab')  
    pyautogui.press('tab')
    pyautogui.press('enter')

print("Processing completed or stopped. All progress has been saved.")


Index(['PYTHON', 'THAI', 'SEEKER', 'CORRECT?', 'caseid', 'Age', 'AgeDay',
       'Sex', 'Disc Type', 'LOSD', 'LOSHr', 'PDx', 'SDx1', 'SDx2', 'SDx3',
       'SDx4', 'SDx5', 'SDx6', 'SDx7', 'SDx8', 'SDx9', 'SDx10', 'SDx11',
       'SDx12', 'Proc1', 'Proc2', 'Proc3', 'Proc4', 'Proc5', 'Proc6', 'Proc7',
       'Proc8', 'Proc9', 'Proc10', 'Proc11', 'Proc12', 'Proc13', 'Proc14',
       'Proc15', 'Proc16', 'Proc17', 'Proc18', 'Proc19', 'Proc20'],
      dtype='object')


NameError: name 'proc_value' is not defined